In [ ]:
# ===== NB06 | CELL 1: probabilities from your saved baseline models =====
# WHAT: load your saved TF-IDF vectorisers + A1 (Logistic Regression) + A2 (calibrated SVM) and predict
#       validation, internal test and external test. Saved to results/predictions/.
# WHY:  notebook 03 saved the models but not the probabilities. Thresholds, the ensemble and the final
#       tables all need them. Test predictions are SAVED here but NOT looked at until the freeze.
from google.colab import drive
drive.mount('/content/drive')
import sys, importlib, time, joblib
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
sys.path.insert(0, f'{PROJECT_DIR}/src')
import toxlib; importlib.reload(toxlib)
from toxlib import LABELS, SPLITS, load_split, save_probs
import numpy as np, pandas as pd, scipy.sparse as sp
CK = f'{PROJECT_DIR}/models/saved_checkpoints'

word = joblib.load(f'{CK}/word_tfidf_vectorizer.pkl')
char = joblib.load(f'{CK}/char_tfidf_vectorizer.pkl')
models = {'A1': joblib.load(f'{CK}/A1_logistic_regression.pkl'),
          'A2': joblib.load(f'{CK}/A2_calibrated_linear_svm.pkl')}

def predict_chunks(df, model, chunk=8000):          # chunks keep memory low on the big external set
    out = []
    for s in range(0, len(df), chunk):
        txt = df.comment_text.iloc[s:s + chunk].fillna('')
        X = sp.hstack([word.transform(txt), char.transform(txt)]).tocsr()
        out.append(model.predict_proba(X))
    return np.vstack(out)

t0 = time.time()
for split in SPLITS:
    df = load_split(PROJECT_DIR, split)
    for run, m in models.items():
        P = predict_chunks(df, m)
        save_probs(PROJECT_DIR, run, split, P)
        print(f'{run} {split:14s} saved {P.shape}  ({time.time() - t0:.0f}s)')
print('CELL 1 DONE')

In [ ]:
# ===== NB06 | CELL 2: fair baseline comparison (validation only) =====
# WHAT: tune one threshold per label on validation, then compare A1 and A2 by AP and by F1 at 0.5 and tuned.
# WHY:  notebook 03 compared F1 at a flat 0.5, which unfairly favoured the SVM. AP and tuned F1 are fair.
import sys, importlib, json
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
sys.path.insert(0, f'{PROJECT_DIR}/src')
import toxlib; importlib.reload(toxlib)
from toxlib import LABELS, load_split, load_probs, macro_ap, tune_thresholds, per_label_report, log_run
import numpy as np, pandas as pd

Yva = load_split(PROJECT_DIR, 'val')[LABELS].values
names = {'A1': 'TF-IDF word+char, Logistic Regression (no class weights)',
         'A2': 'TF-IDF word+char, calibrated Linear SVM (no class weights)'}
rows, thresholds = [], {}
for run, desc in names.items():
    P = load_probs(PROJECT_DIR, run, 'val')
    thr = tune_thresholds(Yva, P)
    thresholds[run] = thr
    flat = per_label_report(Yva, P, 0.5); tuned = per_label_report(Yva, P, thr)
    print(f'\n{run}: per-label validation results with TUNED thresholds'); print(tuned.to_string(index=False))
    tuned.to_csv(f'{PROJECT_DIR}/results/tables/{run}_val_per_label_tuned.csv', index=False)
    rows.append({'run': run, 'Macro AP': round(macro_ap(Yva, P), 4),
                 'Macro F1 @0.5': round(flat.F1.mean(), 4), 'Macro F1 tuned': round(tuned.F1.mean(), 4)})
    log_run(PROJECT_DIR, run, model=desc, max_len='-', loss='-', seed=42,
            val_macro_AP=round(macro_ap(Yva, P), 4), val_tuned_macro_F1=round(tuned.F1.mean(), 4))
T = pd.DataFrame(rows)
T.to_csv(f'{PROJECT_DIR}/results/tables/T07_baseline_fair_comparison.csv', index=False)
json.dump(thresholds, open(f'{PROJECT_DIR}/results/experiment_logs/baseline_val_thresholds.json', 'w'), indent=2)
print('\nT07 baseline comparison (validation):'); print(T.to_string(index=False))
print('CELL 2 DONE')